# Lab Assignment 3
### MPATE-GE 2598 Fundamentals of Digital Signal Theory Lab

In this lab assignment, you will synthesize signals using wave generators and plot their waveforms.

## Background

**Sine Wave**:
$y(t) = \sin(\omega_0 t + \phi)$

Complex periodic waveforms can be constructed by adding a series of sine waves of frequencies that are mathematically related to each other.

The summation formulae for square, triangle, and sawtooth waves are:

**Square Wave**:

$y(t) = \sum_{k=1}^{k_{max}} \frac{1}{k} \sin(k\omega_0t)$ where $k$ is odd.

**Triangle Wave**:

$y(t) = \sum_{k=1}^{k_{max}} (-1)^{(k-1)/2} \frac{1}{k^2} \sin(k\omega_0t)$ where $k$ is odd.

**Sawtooth Wave**:

$y(t) = - \sum_{k=1}^{k_{max}} \frac{1}{k} \sin(k\omega_0t)$

**Note**: $\omega_0 = 2\pi f_0$

**Import Libraries**

In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio
import matplotlib.pyplot as plt

## Part 1: Signal Generation

Fill in the skeleton code functions to address the tasks marked in each of them.


**Note**:
When selecting odd overtones, make sure you process the correct amount of odd overtones corresponding to `num_overtones` instead of stopping the function at the overtone count equal to `num_overtones`. 

For example, if we want 3 odd overtones, we need to stop the loop at overtone 7 ($f_0 + f_3 + f_5 + f_7$), not at overtone 3 ($f_0 + f_3$).

### 1.1 `sinewave()`

The function below is incomplete. Fill in the missing lines of code to generate the `time_vector` and `signal`.

In [ ]:
def sinewave(fs, duration, f0, phi):
    '''Generate a sine wave.

    Parameters
    ----------
    fs : The sampling rate (in Hz)

    duration : The desired length of the
               output signal (in seconds)

    f0 : The fundamental frequency (in Hz)

    phi : The phase of the signal (in radians)
        
    Returns
    -------
    time_vector : of type np.ndarray
                  The timestamps of each sample

    signal : of type np.ndarray
             The synthesized sine wave
    '''
    # timestamps spaced one sampling period (1/fs seconds) apart
    time_vector = np.arange(0, duration, 1/fs)
    
    # y(t) = sin(2*pi*f0*t + phi)
    signal = np.sin(2 * np.pi * f0 * time_vector + phi)
    
    return time_vector, signal

### 1.2 `cosinewave()`

Copy and paste the code from your `sinewave()` function here. Modify it to generate a cosine wave **without using** `np.cos()`. You may only use `np.sin()`.

In [ ]:
def cosinewave(fs, duration, f0, phi):
    '''Generate a cosine wave.

    Parameters
    ----------
    fs : The sampling rate (in Hz)

    duration : The desired length of the
               output signal (in seconds)

    f0 : The fundamental frequency (in Hz)
    
    phi : The phase of the signal (in radians)
        
    Returns
    -------
    time_vector : of type np.ndarray
                  The timestamps of each sample

    signal : of type np.ndarray
             The synthesized cosine wave
    '''
    time_vector = np.arange(0, duration, 1/fs)
    
    # cos(x) = sin(x + pi/2): a cosine is a sine shifted a quarter cycle
    signal = np.sin(2 * np.pi * f0 * time_vector + phi + np.pi/2)

    return time_vector, signal

### 1.3 `squarewave()`

The code for generating a square wave has been provided for you. Modify the function so that no aliased overtones are present in the output signal. Add comments to your function to explain what your code is doing to address aliasing.

In [ ]:
def squarewave(fs, duration, f0, num_overtones):
    '''Generate a square wave.

    Parameters
    ----------
    fs : The sampling rate (in Hz)

    duration : The desired length of the
               output signal (in seconds)

    f0 : The fundamental frequency (in Hz)

    num_overtones : The number of odd overtones
        
    Returns
    -------
    time_vector : of type np.ndarray
                  The timestamps of each sample

    signal : of type np.ndarray
             The synthesized square wave
    '''
    time_vector = np.arange(0, duration, 1/fs)
    signal = np.zeros(len(time_vector))
    
    # Nyquist frequency: the highest frequency we can represent at fs
    nyquist = fs / 2
    
    for k in range(1, 2 * (num_overtones + 1), 2):
        # The k-th harmonic has frequency k * f0. If it is at or above
        # Nyquist it would alias (fold back to a false lower frequency).
        # Harmonics only get higher as k grows, so once one is too high,
        # all the remaining ones are too, and we can stop the loop.
        if k * f0 >= nyquist:
            break
        signal += np.sin(k * 2 * np.pi * f0 * time_vector) / k
    
    return time_vector, signal

### 1.4 `triangularwave()`

(a) The provided code is **not** functional. Address the problems so that the function operates correctly. Do not rewrite the entire function; the basic functionality of the code must remain the same.

In [ ]:
def triangularwave(fs, duration, f0, num_overtones):
    '''Generate a triangular wave.

    Parameters
    ----------
    fs : The sampling rate (in Hz)

    duration : The desired length of the
               output signal (in seconds)

    f0 : The fundamental frequency (in Hz)

    num_overtones : The number of odd overtones
        
    Returns
    -------
    time_vector : of type np.ndarray
                  The timestamps of each sample

    signal : of type np.ndarray
             The synthesized triangular wave
    '''
    time_vector = np.arange(0, duration, 1/fs)
    signal = np.zeros(len(time_vector))      # FIX 1: was np.arange(duration)
    
    i = 0                                    # FIX 2: was i = 1
    k = 1
    
    while i <= num_overtones:                # FIX 2: was i < num_overtones
        signal += ((-1)**((k-1)//2)) * np.sin(2 * np.pi * f0 * k * time_vector) / k**2   # FIX 3: was =
        i += 1                               # FIX 4: was i += 2
        k += 2                               # FIX 4: was k += 1
        
    return time_vector, signal 

(b) Discuss what changes you made and why they were necessary in context of the triangle wave.

**Answer**:

1. **`signal = np.arange(duration)` → `np.zeros(len(time_vector))`.** The original array had the wrong length (`duration` elements instead of one per sample) and was not filled with zeros. The signal needs to start as silence with the same length as `time_vector` so the harmonics can be added onto it.

2. **Loop counter: `i = 1`, `while i < num_overtones` → `i = 0`, `while i <= num_overtones`.** `i` counts how many sine terms have been added. We need the fundamental plus `num_overtones` odd overtones, which is `num_overtones + 1` terms in total. The original condition stopped too early.

3. **`signal = ...` → `signal += ...`.** The original line overwrote the signal on every pass, so only the last harmonic survived. A triangle wave is a *sum* of harmonics, so each one must be added to the running total.

4. **`i += 2`, `k += 1` → `i += 1`, `k += 2`.** The two increments were swapped. A triangle wave contains only **odd** harmonics, so `k` must step 1, 3, 5, 7, ... With `k += 1`, even harmonics were included, and for even `k` the sign term `(-1)**((k-1)/2)` has a non-integer exponent. Meanwhile `i` counts terms, so it should step by 1. I also changed `/2` to `//2` so the exponent is an integer, which makes the alternating sign (+, −, +, −, ...) explicit.

### 1.5 `sawtoothwave()`

You have been provided the function for generating a sawtooth wave.

In [ ]:
def sawtoothwave(fs, duration, f0, num_overtones):
    '''Generate a sawtooth wave.

    Parameters
    ----------
    fs : The sampling rate (in Hz)

    duration : The desired length of the
               output signal (in seconds)

    f0 : The fundamental frequency (in Hz)

    num_overtones : The number of overtones
        
    Returns
    -------
    time_vector : of type np.ndarray
                  The timestamps of each sample

    signal : of type np.ndarray
             The synthesized sawtooth wave
    '''
    time_vector = np.arange(0, duration, 1/fs)
    
    signal = np.zeros(duration * fs)

    # (b) while loop replaced with a for loop:
    # k = 1, 2, ..., num_overtones + 1 (fundamental + all overtones)
    for k in range(1, num_overtones + 2):
        signal -= np.sin(2 * np.pi * f0 * k * time_vector) / k

    return time_vector, signal

(a) Answer the following questions about the function.

(i) Why is the hop / step size for the time vector `1/fs`?

**Answer**: Samples are taken every sampling period, T = 1/fs seconds. With fs samples per second, consecutive samples are 1/fs seconds apart, so stepping the time vector by 1/fs gives exactly one timestamp per sample.

(ii) What are the units of `duration * sr`?

**Answer**: Samples. Duration is in seconds and the sampling rate is in samples per second, so seconds × samples/second = samples. This is the total number of samples in the signal.

(iii) Why is the `while` loop `<=` and why does it require a `+ 1`?

**Answer**: `k` starts at 1, which is the fundamental, not an overtone. To include the fundamental plus `num_overtones` overtones, `k` must reach `num_overtones + 1`. The `<=` makes that last value included; with `<` the loop would stop one harmonic short.

(iv) The formula includes a negative sign in the summation. What would be the result if the sign was removed?

**Answer**: The waveform would be flipped upside down (inverted in polarity, a 180° phase shift). With the negative sign the ramp rises and then drops sharply. Without it the ramp falls and then jumps up (a reverse sawtooth). It contains the same harmonics with the same magnitudes, so it sounds the same.

(b) Return to the function and replace the `while` loop with a `for` loop.

## Part 2: Plot and Playback

You should utilize **each function** above to generate and display one of each of the signal types. Each generated signal should be played back using the `Audio` function. Remember to use the correct sampling rate during playback. You may choose any inputs that you would like.

The plots must show amplitude vs. time (not samples). In order to make the waveforms visible and clear, **plot only four (4) periods of each signal**. Make sure you label your plot.

**Important**: Use `plt.xlim()` and the relationship between period and the fundamental frequency to limit the time axis; do not hard code it. If a different frequency is selected, the graph must also display four cycles of the waveform.

### 2.1 Sine Wave

In [ ]:
fs = 44100
duration = 1
f0 = 220
t, signal = sinewave(fs, duration, f0, phi=0)

T = 1 / f0                       # period in seconds
plt.plot(t, signal)
plt.xlim(0, 4 * T)               # show exactly 4 periods
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"{f0} Hz Sine Wave")
plt.show()

Audio(signal, rate=fs)

### 2.2 Cosine Wave

In [ ]:
fs = 44100
duration = 1
f0 = 220
t, signal = cosinewave(fs, duration, f0, phi=0)

T = 1 / f0                       # period in seconds
plt.plot(t, signal)
plt.xlim(0, 4 * T)               # show exactly 4 periods
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"{f0} Hz Cosine Wave")
plt.show()

Audio(signal, rate=fs)

### 2.3 Square Wave

In [ ]:
fs = 44100
duration = 1
f0 = 220
t, signal = squarewave(fs, duration, f0, num_overtones=20)

T = 1 / f0                       # period in seconds
plt.plot(t, signal)
plt.xlim(0, 4 * T)               # show exactly 4 periods
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"{f0} Hz Square Wave (20 odd overtones)")
plt.show()

Audio(signal, rate=fs)

### 2.4 Triangle Wave

In [ ]:
fs = 44100
duration = 1
f0 = 220
t, signal = triangularwave(fs, duration, f0, num_overtones=20)

T = 1 / f0                       # period in seconds
plt.plot(t, signal)
plt.xlim(0, 4 * T)               # show exactly 4 periods
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"{f0} Hz Triangle Wave (20 odd overtones)")
plt.show()

Audio(signal, rate=fs)

### 2.5 Sawtooth Wave

In [ ]:
fs = 44100
duration = 1
f0 = 220
t, signal = sawtoothwave(fs, duration, f0, num_overtones=40)

T = 1 / f0                       # period in seconds
plt.plot(t, signal)
plt.xlim(0, 4 * T)               # show exactly 4 periods
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"{f0} Hz Sawtooth Wave (40 overtones)")
plt.show()

Audio(signal, rate=fs)

## Part 3: Bonus Exercise

A [**pulse wave**](https://en.wikipedia.org/wiki/Pulse_wave) (or pulse train) is a periodic, non-sinusoidal waveform which alternates between a high value representing "on" (e.g., 1) and a low value representing "off" (e.g., 0). The amount of time it spends in the high vs. low state in a single period varies according to its **duty cycle** (a ratio).

You have been provided the skeleton of a pulse wave generator below. Complete the function and include comments explaining the logic your code.

**Note**: You may **not** use an existing signal generator in your function.

In [ ]:
def pulsewave(fs, duration, f, duty):
    '''Generate a pulse wave which oscillates between 0 and 1.

    Parameters
    ----------
    fs : The sampling rate (in Hz)

    duration : The desired length of the
               output signal (in seconds)

    f : The frequency of pulses (in Hz)
    
    duty : The proportion of the period during which
           the signal is "on" (0 ≤ duty ≤ 1)
        
    Returns
    -------
    time_vector : of type np.ndarray
                  The timestamps of each sample

    signal : of type np.ndarray
             The synthesized pulse wave
    '''
    # one timestamp per sample
    time_vector = np.arange(0, duration, 1/fs)
    
    # Position within the current period, as a fraction from 0 to 1:
    # t * f counts how many cycles have elapsed; % 1 keeps only the
    # fractional part, so it ramps 0 -> 1 and resets every period.
    position = (time_vector * f) % 1
    
    # The signal is "on" (1) for the first `duty` fraction of each
    # period and "off" (0) for the rest.
    signal = (position < duty).astype(float)

    return time_vector, signal

Now, generate a pulse wave with input parameters of your choice, play it back, and plot it so that the waveform is visible.

In [ ]:
fs = 44100
duration = 1
f = 220
duty = 0.25

t, signal = pulsewave(fs, duration, f, duty)

T = 1 / f
plt.plot(t, signal)
plt.xlim(0, 4 * T)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title(f"{f} Hz Pulse Wave (duty = {duty})")
plt.show()

Audio(signal, rate=fs)

---
_End of Lab 4_